# 4.1 Inférence — modèle 1024×1024

Même pipeline qu'en section 4, appliqué au modèle réentraîné à 1024×1024 (`yolov8m_epi_1024-2/weights/best.pt`, epoch 79).

**Motivation (voir doc 05)** : le modèle 640×640 produisait des fausses alertes sur des frames où casques et gilets étaient clairement visibles, à cause d'une confiance de détection instable sur les petits EPI. Le modèle 1024×1024 (mAP50 val : 0.674 vs 0.645, Recall : 0.653 vs 0.609) devrait réduire ce phénomène.

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import cv2
import numpy as np
import yaml
import torch

DATASET_ROOT = Path("/root/Projet_Image/SH17dataset")
RUNS_DIR     = Path("/root/Projet_Image/runs")
BEST_WEIGHTS = RUNS_DIR / 'yolov8m_epi_1024-2' / 'weights' / 'best.pt'
OUTPUT_DIR   = Path("/root/Projet_Image/inference_output_1024")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE       = 0 if torch.cuda.is_available() else 'cpu'

model = YOLO(str(BEST_WEIGHTS))
print(f"Modèle chargé : {BEST_WEIGHTS}")
print(f"Device        : {DEVICE}")
print(f"Classes       : {model.names}")

## 1. Règle de non-conformité

Identique à 4 — voir doc 05 pour le détail.

In [ ]:
REGLES_CONFORMITE = {
    'helmet':      {'partie_corps': 'head',   'message': 'casque manquant'},
    'safety-vest': {'partie_corps': 'person', 'message': 'gilet de securite manquant'},
    'gloves':      {'partie_corps': 'hands',  'message': 'gants manquants'},
}

def iou(boite_a, boite_b):
    """Taux de chevauchement entre deux boîtes (x1, y1, x2, y2) — voir doc 03 pour la définition de l'IoU."""
    xa1, ya1, xa2, ya2 = boite_a
    xb1, yb1, xb2, yb2 = boite_b
    x1, y1 = max(xa1, xb1), max(ya1, yb1)
    x2, y2 = min(xa2, xb2), min(ya2, yb2)
    inter  = max(0, x2 - x1) * max(0, y2 - y1)
    aire_a = (xa2 - xa1) * (ya2 - ya1)
    aire_b = (xb2 - xb1) * (yb2 - yb1)
    union  = aire_a + aire_b - inter
    return inter / union if union > 0 else 0.0

def verifier_conformite(detections, seuil_iou=0.1):
    """Applique REGLES_CONFORMITE aux détections brutes. Renvoie (non_conforme, motifs)."""
    motifs = []
    for classe_epi, regle in REGLES_CONFORMITE.items():
        boites_partie = [d['boite'] for d in detections if d['classe'] == regle['partie_corps']]
        boites_epi    = [d['boite'] for d in detections if d['classe'] == classe_epi]
        for boite_partie in boites_partie:
            protege = any(iou(boite_partie, boite_epi) > seuil_iou for boite_epi in boites_epi)
            if not protege:
                motifs.append(regle['message'])
    return len(motifs) > 0, motifs

## 2. Incrustation visuelle (overlay)

Identique à 4.

In [ ]:
def incruster_resultats(image, detections, non_conforme, motifs, conf_min=0.4):
    img = image.copy()
    h, w = img.shape[:2]

    for det in detections:
        if det['confiance'] < conf_min:
            continue
        x1, y1, x2, y2 = map(int, det['boite'])
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 200, 0), 2)
        cv2.putText(img, f"{det['classe']} {det['confiance']:.2f}", (x1, max(y1 - 6, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 0), 1)

    if non_conforme:
        cv2.rectangle(img, (0, 0), (w, 36), (0, 0, 255), -1)
        texte = "NON CONFORME : " + ", ".join(motifs)
        cv2.putText(img, texte, (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    return img

In [ ]:
def detecter_et_verifier(image, conf_min=0.4, seuil_iou=0.1):
    """Détection + règle de conformité, sans incrustation. Renvoie (detections, non_conforme, motifs)."""
    resultat = model.predict(image, conf=conf_min, device=DEVICE, verbose=False)[0]
    detections = [
        {'classe': model.names[int(c)], 'boite': boite.tolist(), 'confiance': float(conf)}
        for boite, c, conf in zip(resultat.boxes.xyxy.cpu().numpy(),
                                   resultat.boxes.cls.cpu().numpy(),
                                   resultat.boxes.conf.cpu().numpy())
    ]
    non_conforme, motifs = verifier_conformite(detections, seuil_iou)
    return detections, non_conforme, motifs


def analyser_image(image, conf_min=0.4, seuil_iou=0.1):
    """Pipeline complet : détection -> règle de conformité -> incrustation."""
    detections, non_conforme, motifs = detecter_et_verifier(image, conf_min, seuil_iou)
    image_incrustee = incruster_resultats(image, detections, non_conforme, motifs, conf_min)
    return image_incrustee, non_conforme, motifs

## 3. Test sur les images du split test

In [ ]:
with open(DATASET_ROOT / 'sh17.yaml') as f:
    config_dataset = yaml.safe_load(f)

TEST_IMAGES_DIR   = DATASET_ROOT / config_dataset['test']
OUTPUT_IMAGES_DIR = OUTPUT_DIR / 'images'
OUTPUT_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

chemins = sorted(TEST_IMAGES_DIR.glob('*.jpg')) + sorted(TEST_IMAGES_DIR.glob('*.jpeg'))
print(f"{len(chemins)} images de test trouvées dans {TEST_IMAGES_DIR}")

n_non_conformes = 0
for chemin in chemins:
    image = cv2.imread(str(chemin))
    image_incrustee, non_conforme, motifs = analyser_image(image)
    cv2.imwrite(str(OUTPUT_IMAGES_DIR / chemin.name), image_incrustee)
    if non_conforme:
        n_non_conformes += 1
        print(f"  {chemin.name} -> NON CONFORME ({', '.join(motifs)})")

print(f"\n{n_non_conformes}/{len(chemins)} images marquées non conformes — résultats dans {OUTPUT_IMAGES_DIR}")

## 4. Pipeline vidéo — avec lissage temporel

Identique à 4 (fenêtre glissante de 15 frames, seuil 50%). La vidéo source est réutilisée depuis `inference_output/`.

In [ ]:
from collections import deque

def analyser_video(chemin_entree, chemin_sortie, conf_min=0.4, seuil_iou=0.1, fenetre=15, seuil_alerte=0.5):
    """Comme analyser_image, mais lisse le verdict dans le temps."""
    cap = cv2.VideoCapture(str(chemin_entree))
    largeur = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    hauteur = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps     = cap.get(cv2.CAP_PROP_FPS)

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(str(chemin_sortie), fourcc, fps, (largeur, hauteur))

    historique = deque(maxlen=fenetre)
    n_frames, n_non_conformes = 0, 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        detections, non_conforme_brut, motifs = detecter_et_verifier(frame, conf_min, seuil_iou)
        historique.append(non_conforme_brut)

        non_conforme = (sum(historique) / len(historique)) >= seuil_alerte
        motifs_affiches = motifs if non_conforme else []

        frame_incrustee = incruster_resultats(frame, detections, non_conforme, motifs_affiches, conf_min)
        writer.write(frame_incrustee)
        n_frames += 1
        n_non_conformes += int(non_conforme)

    cap.release()
    writer.release()
    print(f"{n_frames} frames traitées, {n_non_conformes} marquées non conformes (verdict lissé sur {fenetre} frames, seuil {seuil_alerte:.0%})")
    print(f"Vidéo annotée sauvegardée : {chemin_sortie}")

In [ ]:
VIDEO_SOURCE = Path("/root/Projet_Image/inference_output") / 'video_source.mp4'
VIDEO_SORTIE = OUTPUT_DIR / 'video_annotee_1024.mp4'
analyser_video(VIDEO_SOURCE, VIDEO_SORTIE)

## 5. Comparaison : sans lissage temporel

In [ ]:
VIDEO_SORTIE_BRUT = OUTPUT_DIR / 'video_annotee_1024_sans_lissage.mp4'
analyser_video(VIDEO_SOURCE, VIDEO_SORTIE_BRUT, fenetre=1)